In [1]:
# Getting imports and the model builder
# Combines what used to be 12 separate per-(model, window_size) notebooks into one
# parameterized notebook that loops over every combination.
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")
sys.path.insert(0, "../baseline")

from smolagents import OpenAIModel
from dotenv import load_dotenv
load_dotenv()

MODEL_NAMES = ["gpt-4o", "gpt-5.4-mini", "Qwen/Qwen3.7-Plus", "Qwen/Qwen3.5-9B"]
WINDOW_SIZES = [1, 3, 5]


def build_model(model_name: str) -> OpenAIModel:
    if model_name in ["gpt-4o", "gpt-5.4-mini"]:
        return OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])
    # Together requires enable_thinking nested inside chat_template_kwargs for vLLM-served
    # open-weight models -- verified this works for both Qwen3.7-Plus and Qwen3.5-9B.
    # client_kwargs timeout bounds a single API call so a stalled/hanging stream fails within
    # 5 minutes instead of hanging indefinitely -- evaluate_agent already catches such errors.
    return OpenAIModel(
        model_id=model_name,
        api_base="https://api.together.ai/v1/",
        api_key=os.environ["TOGETHER_API_KEY"],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        client_kwargs={"timeout": 300.0},
    )

In [2]:
# Getting the tools setup and the MarkovReActCodeAgent builder
from smolagents.monitoring import LogLevel
from common_setup import assert_no_reasoning, build_tools
from smolagents.markov_react import MarkovReActCodeAgent


def build_agent(model, model_name: str, window_size: int, tools):
    return MarkovReActCodeAgent(
        tools=tools,
        model=model,
        max_steps=50,
        verbosity_level=LogLevel.DEBUG,
        additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
        window_size=window_size,
        stream_outputs="Qwen" in model_name,
        step_callbacks=[assert_no_reasoning] if "Qwen" in model_name else None,
    )

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [3]:
# Load GAIA validation set from HuggingFace
# Visit https://huggingface.co/datasets/gaia-benchmark/GAIA to request access first
import pandas as pd
from common_setup import evaluate_agent, load_gaia_dataset, question_scorer

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")
print(pd.DataFrame(eval_ds)["task"].value_counts())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples
task
2    86
1    53
3    26
Name: count, dtype: int64


In [4]:
# One model (and its tools) is built once, then reused across all 3 window sizes.
# pickle_dir naming matches the original 12 notebooks exactly, so every already-computed
# result is picked up from cache -- this loop makes zero new API calls on a fresh run.
all_results = {}

for model_name in MODEL_NAMES:
    model = build_model(model_name)
    tools, ti_tool, visualizer = build_tools(model)

    for window_size in WINDOW_SIZES:
        agent = build_agent(model, model_name, window_size, tools)
        pickle_dir = f"markov_react_{model_name}_w{window_size}"
        print(f"\n=== {model_name}, window_size={window_size} ===")
        results = evaluate_agent(
            agent,
            eval_ds,
            ti_tool,
            visualizer,
            output_file=f"{pickle_dir}.jsonl",
            pickle_dir=pickle_dir,
        )
        all_results[(model_name, window_size)] = results


=== gpt-4o, window_size=1 ===

[1/165] (cached) A paper about AI regulation that was originally submitted to arXiv.org in June 2022 shows a figure w...
  ✓ | cached

[2/165] (cached) I’m researching species that became invasive after people who kept them as pets released them. There...
  ✓ | cached

[3/165] (cached) If we assume all articles published by Nature in 2020 (articles, only, not book reviews/columns, etc...
  ✗ | cached

[4/165] (cached) In Unlambda, what exact charcter or text needs to be added to correct the following code to output "...
  ✗ | cached

[5/165] (cached) If Eliud Kipchoge could maintain his record-making marathon pace indefinitely, how many thousand hou...
  ✗ | cached

[6/165] (cached) The attached spreadsheet shows the inventory for a movie and video game rental store in Seattle, Was...
  ✓ | cached

[7/165] (cached) How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use...
  ✓ | cached

[8/165] (cached) The ob


[150/165] (cached) Take the gender split from the 2011 Bulgarian census about those who have completed tertiary educati...
  ✗ | cached

[151/165] (cached) What was the actual enrollment count of the clinical trial on H. pylori in acne vulgaris patients fr...
  ✓ | cached

[152/165] (cached) I'd like to learn more about some popular reality television competition shows. As of the end of the...
  ✗ | cached

[153/165] (cached) Where were the Vietnamese specimens described by Kuznetzov in Nedoshivina's 2010 paper eventually de...
  ✗ | cached

[154/165] (cached) A standard Rubik’s cube has been broken into cubes making up its sides. The cubes are jumbled, and o...
  ✗ | cached

[155/165] (cached) What country had the least number of athletes at the 1928 Summer Olympics? If there's a tie for a nu...
  ✓ | cached

[156/165] (cached) I read a paper about multiwavelength observations of fast radio bursts back in March 2021 on Arxiv, ...
  ✗ | cached

[157/165] (cached) Who are the pitchers 


[157/165] (cached) Who are the pitchers with the number before and after Taishō Tamai's number as of July 2023? Give th...
  ✗ | cached

[158/165] (cached) The attached Excel file contains the sales of menu items for a local fast-food chain. What were the ...
  ✓ | cached

[159/165] (cached) What is the first name of the only Malko Competition recipient from the 20th Century (after 1977) wh...
  ✗ | cached

[160/165] (cached) In the YouTube 360 VR video from March 2018 narrated by the voice actor of Lord of the Rings' Gollum...
  ✗ | cached

[161/165] (cached) In NASA's Astronomy Picture of the Day on 2006 January 21, two astronauts are visible, with one appe...
  ✗ | cached

[162/165] (cached) In the film Goldfinger, what color was the object that James Bond concealed himself and his companio...
  ✓ | cached

[163/165] (cached) As of May 2023, how many stops are between South Station and Windsor Gardens on MBTA’s Franklin-Foxb...
  ✓ | cached

[164/165] (cached) In the 2015 Metropoli


[142/165] (cached) What is the absolute difference in tens of thousands between the population of chinstrap penguins on...
  ✓ | cached

[143/165] (cached) The attached file lists the locomotives owned by a local railroad museum. It gives each locomotive’s...
  ✓ | cached

[144/165] (cached) Hi, I was out sick from my classes on Friday, so I'm trying to figure out what I need to study for m...
  ✗ | cached

[145/165] (cached) When was a picture of St. Thomas Aquinas first added to the Wikipedia page on the Principle of doubl...
  ✗ | cached

[146/165] (cached) A 5-man group made up of one tank, one healer, and three DPS is doing a dungeon that was just releas...
  ✗ | cached

[147/165] (cached) On June 6, 2023, an article by Carolyn Collins Petersen was published in Universe Today. This articl...
  ✓ | cached

[148/165] (cached) According to Openreview.net, at the NeurIPS 2022 Conference, how many papers by an author named Yuri...
  ✗ | cached

[149/165] (cached) If this whole pint is


[145/165] (cached) When was a picture of St. Thomas Aquinas first added to the Wikipedia page on the Principle of doubl...
  ✗ | cached

[146/165] (cached) A 5-man group made up of one tank, one healer, and three DPS is doing a dungeon that was just releas...
  ✓ | cached

[147/165] (cached) On June 6, 2023, an article by Carolyn Collins Petersen was published in Universe Today. This articl...
  ✓ | cached

[148/165] (cached) According to Openreview.net, at the NeurIPS 2022 Conference, how many papers by an author named Yuri...
  ✗ | cached

[149/165] (cached) If this whole pint is made up of ice cream, how many percent above or below the US federal standards...
  ✗ | cached

[150/165] (cached) Take the gender split from the 2011 Bulgarian census about those who have completed tertiary educati...
  ✗ | cached

[151/165] (cached) What was the actual enrollment count of the clinical trial on H. pylori in acne vulgaris patients fr...
  ✗ | cached

[152/165] (cached) I'd like to learn mor


[29/165] (cached) An office held a Secret Santa gift exchange where each of its twelve employees was assigned one othe...
  ✓ | cached

[30/165] (cached) What is the maximum length in meters of #9 in the first National Geographic short on YouTube that wa...
  ✗ | cached

[31/165] (cached) What two-word type of model did Manash Pratim Kashyap's and PS Fader's studies in customer retention...
  ✓ | cached

[32/165] (cached) What animals that were mentioned in both Ilias Lagkouvardos's and Olga Tapia's papers on the alvei s...
  ✗ | cached

[33/165] (cached) How many High Energy Physics - Lattice articles listed in January 2020 on Arxiv had ps versions avai...
  ✗ | cached

[34/165] (cached) The photograph in the Whitney Museum of American Art's collection with accession number 2022.128 sho...
  ✓ | cached

[35/165] (cached) .rewsna eht sa "tfel" drow eht fo etisoppo eht etirw ,ecnetnes siht dnatsrednu uoy fI...
  ✓ | cached

[36/165] (cached) What is the minimum number of page links a p


[156/165] (cached) I read a paper about multiwavelength observations of fast radio bursts back in March 2021 on Arxiv, ...
  ✗ | cached

[157/165] (cached) Who are the pitchers with the number before and after Taishō Tamai's number as of July 2023? Give th...
  ✓ | cached

[158/165] (cached) The attached Excel file contains the sales of menu items for a local fast-food chain. What were the ...
  ✓ | cached

[159/165] (cached) What is the first name of the only Malko Competition recipient from the 20th Century (after 1977) wh...
  ✓ | cached

[160/165] (cached) In the YouTube 360 VR video from March 2018 narrated by the voice actor of Lord of the Rings' Gollum...
  ✗ | cached

[161/165] (cached) In NASA's Astronomy Picture of the Day on 2006 January 21, two astronauts are visible, with one appe...
  ✗ | cached

[162/165] (cached) In the film Goldfinger, what color was the object that James Bond concealed himself and his companio...
  ✓ | cached

[163/165] (cached) As of May 2023, how m


[55/165] (cached) According to Google Finance, when was the first year the Apple stock went above $50 (without adjusti...
  ✓ | cached

[56/165] (cached) Review the chess position provided in the image. It is black's turn. Provide the correct next move f...
  ✗ | cached

[57/165] (cached) According to Box Office Mojo's 2020 Worldwide Box Office list, how many of the top 10 highest-grossi...
  ✗ | cached

[58/165] (cached) In the year 2022, and before December, what does "R" stand for in the three core policies of the typ...
  ✓ | cached

[59/165] (cached) Who nominated the only Featured Article on English Wikipedia about a dinosaur that was promoted in N...
  ✓ | cached

[60/165] (cached) What writer is quoted by Merriam-Webster for the Word of the Day from June 27, 2022?...
  ✗ | cached

[61/165] (cached) How many pages if the 2023 IPCC report (85 pages version) mentions nuclear energy?...
  ✗ | cached

[62/165] (cached) Given this table defining * on the set S = {a, b, c, d, e}

|*|


[20/165] (cached) Of the authors (First M. Last) that worked on the paper "Pie Menus or Linear Menus, Which Is Better?...
  ✗ | cached

[21/165] (cached) When you take the average of the standard population deviation of the red numbers and the standard s...
  ✗ | cached

[22/165] (cached) Assuming scientists in the famous youtube video The Thinking Machine (Artificial Intelligence in the...
  ✗ | cached

[23/165] (cached) In Series 9, Episode 11 of Doctor Who, the Doctor is trapped inside an ever-shifting maze. What is t...
  ✗ | cached

[24/165] (cached) In terms of geographical distance between capital cities, which 2 countries are the furthest from ea...
  ✗ | cached

[25/165] (cached) In the NCATS PubChem compound database for Food Additive Status classification, find the compound th...
  ✗ | cached

[26/165] (cached) I need to fact-check a citation. This is the citation from the bibliography:

Greetham, David. "Unco...
  ✓ | cached

[27/165] (cached) Which contributor to the vers

In [5]:
rows = []
for (model_name, window_size), results in all_results.items():
    df = pd.DataFrame(results)
    total = len(df)
    correct = df["is_correct"].sum()
    total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0))
    rows.append({
        "Model": model_name,
        "window_size": window_size,
        "Accuracy": f"{correct}/{total} = {correct / total:.1%}",
        "Avg Steps": round(df["num_steps"].mean(), 1),
        "Avg Tokens": round(total_tokens.mean()),
    })

pd.DataFrame(rows)

,Model,window_size,Accuracy,Avg Steps,Avg Tokens
0,gpt-4o,1,47/165 = 28.5%,21.0,83697
1,gpt-4o,3,47/165 = 28.5%,13.4,81128
2,gpt-4o,5,51/165 = 30.9%,11.9,88407
3,gpt-5.4-mini,1,46/165 = 27.9%,8.6,32263
4,gpt-5.4-mini,3,56/165 = 33.9%,5.2,24891
5,gpt-5.4-mini,5,51/165 = 30.9%,5.0,27918
6,Qwen/Qwen3.7-Plus,1,49/165 = 29.7%,32.6,137151
7,Qwen/Qwen3.7-Plus,3,90/165 = 54.5%,24.7,172159
8,Qwen/Qwen3.7-Plus,5,89/165 = 53.9%,22.4,209685
9,Qwen/Qwen3.5-9B,1,48/165 = 29.1%,30.0,133161
